# 02 — Nested evaluation (never fits)

Ensembles on every S1 label set (provider, `L_track(1 s)`, `L_ctrl`, `L_track(3 s)`) and both views; single seeds on provider and `L_ctrl`, both views. Temperature and decoder are re-selected on the six inner validation matches against each label set; the held-out match is scored once. `exp3_ref_ens3` / `exp2_aug` come from exp_3's stored results. Then argmax diagnostics, recall of possessions < 2 s, and the NLL generalisation gap.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp4_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_4')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp4_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp4_protocol as P, exp4_diagnostics as G
for arm in P.ARMS:
    missing = [k for k, *_ in P.P1.job_keys() if not (P.prediction_folder(arm) / f'{k}.npz').exists()]
    assert not missing, f'{arm}: {len(missing)} jobs missing; run 01_fits first'

PREREG verified 254e2e216cb05154 | frozen 2026-11-09T17:09:15+00:00


In [2]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
tasks = [(v, view, ls) for v in P.ENSEMBLE_VARIANTS for ls in C.S1_LABELS for view in C.VIEWS]
tasks += [(arm, view, ls) for arm in P.ARMS for ls in C.DECISION_LABELS for view in C.VIEWS]
t0 = time.time()
with ProcessPoolExecutor(max_workers=4, mp_context=multiprocessing.get_context('fork')) as pool:
    for variant, view, label_set, seconds, tag in pool.map(P.evaluate_and_save, tasks):
        frame = pd.DataFrame(C1.read_json(P.result_path(variant, view, label_set))['rows'])
        print(f'{label_set:8s} {view:12s} {variant:16s} spellF1 {frame.spell_f1.mean():.4f}  max|cnt| {frame.count_error_ratio.abs().max():.3f}  gamma {sorted(set(frame.gamma))}  ({tag} {seconds:.0f}s)', flush=True)
print(f'{len(tasks)} evaluations, {time.time() - t0:.0f}s')

provider original     exp4_full_ens3   spellF1 0.6735  max|cnt| 0.341  gamma [0.0]  (cached 0s)


provider target_like  exp4_full_ens3   spellF1 0.6716  max|cnt| 0.342  gamma [0.0]  (cached 0s)


track_1  original     exp4_full_ens3   spellF1 0.7107  max|cnt| 0.201  gamma [0.0]  (cached 0s)


track_1  target_like  exp4_full_ens3   spellF1 0.7106  max|cnt| 0.228  gamma [0.0]  (cached 0s)


ctrl     original     exp4_full_ens3   spellF1 0.7467  max|cnt| 0.191  gamma [0.0, 0.5]  (cached 0s)


ctrl     target_like  exp4_full_ens3   spellF1 0.7421  max|cnt| 0.208  gamma [0.0, 0.5]  (cached 0s)


... (51 outputs left out)

provider target_like  exp4_aux_s2      spellF1 0.6487  max|cnt| 0.366  gamma [0.0, 0.5]  (cached 0s)


ctrl     original     exp4_aux_s2      spellF1 0.7274  max|cnt| 0.210  gamma [0.0, 0.5]  (cached 0s)


ctrl     target_like  exp4_aux_s2      spellF1 0.7198  max|cnt| 0.210  gamma [0.0, 0.5]  (cached 0s)


60 evaluations, 0s


## Diagnostics: argmax (no decoder) and recall of true possessions < 2 s (provider labels)

In [3]:
cache = {}
variants = (*P.ENSEMBLE_VARIANTS, 'exp3_ref_ens3')
argmax = pd.DataFrame([r for v in variants for ls in C.DECISION_LABELS for view in C.VIEWS
                       for r in G.argmax_rows(v, view, ls, cache)])
argmax.to_csv(C.RESULTS / 'argmax_rows.csv', index=False)
print(argmax.groupby(['label_set', 'view', 'variant'])[['spell_f1', 'count_error_ratio']].mean().round(4).to_string())
recall = pd.DataFrame([r for v in (*variants, 'exp2_aug') for view in C.VIEWS for r in G.short_recall_rows(v, view, cache)])
recall.to_csv(C.RESULTS / 'short_recall.csv', index=False)
print(recall.groupby(['view', 'variant'])[['recall_short_decoded', 'recall_short_argmax', 'recall_all_decoded']].mean().round(4).to_string())

                                              spell_f1  count_error_ratio
label_set view        variant                                            
ctrl      original    exp3_ref_ens3_argmax      0.6355             0.3973
                      exp4_aux_ens3_argmax      0.6322             0.3827
                      exp4_full_ens3_argmax     0.6383             0.3944
                      exp4_touch_ens3_argmax    0.6595             0.3470
          target_like exp3_ref_ens3_argmax      0.6312             0.4016
                      exp4_aux_ens3_argmax      0.6344             0.3717
                      exp4_full_ens3_argmax     0.6377             0.3883
                      exp4_touch_ens3_argmax    0.6507             0.3641
provider  original    exp3_ref_ens3_argmax      0.6475            -0.0716
                      exp4_aux_ens3_argmax      0.6384            -0.0819
                      exp4_full_ens3_argmax     0.6430            -0.0747
                      exp4_touch_ens3_

                             recall_short_decoded  recall_short_argmax  recall_all_decoded
view        variant                                                                       
original    exp2_aug                       0.2887               0.3263              0.6072
            exp3_ref_ens3                  0.3017               0.3481              0.6096
            exp4_aux_ens3                  0.2986               0.3287              0.6066
            exp4_full_ens3                 0.3013               0.3271              0.6102
            exp4_touch_ens3                0.2976               0.3313              0.6177
target_like exp2_aug                       0.2869               0.3225              0.6069
            exp3_ref_ens3                  0.3006               0.3424              0.6095
            exp4_aux_ens3                  0.3017               0.3264              0.6088
            exp4_full_ens3                 0.2980               0.3228              0.6081

## Generalisation gap (provider labels, raw p, T = 1)
Training NLL of every fit (exp_4: logged at fit time; exp_2 recipe: recomputed from its saved weights, inference only) against the held-out NLL of its outer and pair predictions.

In [4]:
t0 = time.time()
train = pd.DataFrame([r for arm in P.ARMS for r in P.train_nll_rows(arm)])
ref_path = C.RESULTS / 'ref_train_nll.csv'
if not ref_path.exists():
    pd.DataFrame([r for arm in P.EXP2_ARMS for r in G.reference_train_nll(arm)]).to_csv(ref_path, index=False)
train = pd.concat([train, pd.read_csv(ref_path)], ignore_index=True)
heldout = pd.DataFrame([r for arm in (*P.ARMS, *P.EXP2_ARMS) for r in G.heldout_nll_rows(arm, cache)])
train.to_csv(C.RESULTS / 'train_nll.csv', index=False)
heldout.to_csv(C.RESULTS / 'heldout_nll.csv', index=False)
gap = G.gap_table(train, heldout)
gap.to_csv(C.RESULTS / 'gap_table.csv', index=False)
print(gap.round(4).to_string(index=False))
print(f'{time.time() - t0:.0f}s')

          arm  train_nll  heldout_nll_original  heldout_nll_target_like  gap_original  gap_target_like      family  gap_growth_vs_exp2_recipe  level flag_gap_growth_gt_25pct
     exp2_aug     0.1165                0.1888                   0.1919        0.0722           0.0754 exp2_recipe                     0.0048    arm                      NaN
  exp2_aug_s1     0.1161                0.1884                   0.1916        0.0723           0.0754 exp2_recipe                     0.0053    arm                      NaN
  exp2_aug_s2     0.1169                0.1881                   0.1903        0.0712           0.0734 exp2_recipe                    -0.0102    arm                      NaN
     exp4_aux     0.1180                0.1856                   0.1885        0.0676           0.0704    exp4_aux                    -0.0597    arm                      NaN
  exp4_aux_s1     0.1174                0.1891                   0.1915        0.0718           0.0741    exp4_aux                